### O Treino do Modelo "Student"
O Racional: O que é o "Student"?
Até agora, usámos um modelo pesado e complexo (o Teacher, RoBERTa) para ler o texto bruto e criar os rótulos com alta confiança. No entanto, em produção (no mundo real), usar um modelo tão pesado para cada nova reclamação seria caro e lento.

Por isso, vamos "destilar" (Knowledge Distillation) esse conhecimento. Vamos treinar um modelo mais leve, rápido e eficiente — o DistilBERT (o Student) — para aprender a prever esses mesmos rótulos, mas lendo apenas o texto limpo (cleaned_text) preparado na Fase 4. Isto cumpre as regras do projeto e cria um modelo altamente otimizado.



[Voltar para notebook principal](./0_notebook_principal.ipynb)

In [1]:
import polars as pl
import torch
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
import evaluate

# --- Configurações e Caminhos ---
BASE_DIR = Path.cwd()
INPUT_FILE = BASE_DIR / "data" / "parquet" / "processed_dataset.parquet"
MODEL_OUTPUT_DIR = BASE_DIR / "models" / "student_model"

def train_student_model():
    print("--- 🎓 FASE 5: Treino do Modelo Student (DistilBERT) ---")
    
    if not INPUT_FILE.exists():
        print(f"❌ Ficheiro não encontrado: {INPUT_FILE}")
        return

    # 1. Carregar Dados
    print("📂 A carregar o dataset processado...")
    df = pl.read_parquet(INPUT_FILE)
    
    # Garantir que só temos as classes 0 e 1 (remover qualquer ruído)
    df = df.filter(pl.col("label_final").is_in([0, 1]))
    
    # Para acelerar o treino no contexto deste projeto (e não esgotar a RAM/VRAM),
    # se o dataset for gigantesco (ex: > 100k linhas), podemos usar uma amostra representativa.
    # Vamos limitar a 50.000 registos bem balanceados (opcional, remova o limit se quiser treinar com tudo e tiver tempo).
    if len(df) > 50000:
        print(f"Aviso: Dataset com {len(df)} registos. A realizar amostragem para 50k para treino rápido...")
        df = df.sample(50000, seed=42)

    texts = df["cleaned_text"].to_list()
    labels = df["label_final"].to_list()

    # 2. Divisão Treino / Validação (80% / 20%)
    print("🔀 A dividir dados em Treino e Validação (Stratified)...")
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        texts, labels, test_size=0.2, stratify=labels, random_state=42
    )

    # 3. Preparar formato Hugging Face Datasets
    train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
    val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})

    # 4. Tokenização (DistilBERT)
    print("🔤 A iniciar Tokenização...")
    MODEL_NAME = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenize_function(examples):
        # max_length=128 é ideal para texto limpo e poupa muita VRAM
        return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

    train_tokenized = train_dataset.map(tokenize_function, batched=True)
    val_tokenized = val_dataset.map(tokenize_function, batched=True)

    # 5. Configurar Métrica de Avaliação (F1-Score e Accuracy)
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")

    def compute_metrics(eval_pred):
        logits, true_labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        acc = accuracy_metric.compute(predictions=predictions, references=true_labels)["accuracy"]
        # Usamos average='macro' porque as classes são desbalanceadas
        f1 = f1_metric.compute(predictions=predictions, references=true_labels, average="macro")["f1"]
        return {"accuracy": acc, "f1_macro": f1}

    # 6. Carregar Modelo
    print(f"🤖 A carregar modelo base ({MODEL_NAME})...")
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    # 7. Argumentos de Treino
    training_args = TrainingArguments(
        output_dir=MODEL_OUTPUT_DIR / "checkpoints",
        eval_strategy="epoch",      # Avaliar no fim de cada época
        save_strategy="epoch",      # Guardar no fim de cada época
        learning_rate=2e-5,         # Taxa de aprendizagem recomendada para fine-tuning
        per_device_train_batch_size=32, # Ajuste para 16 se der "Out of Memory" na GPU
        per_device_eval_batch_size=64,
        num_train_epochs=2,         # 2 épocas são perfeitas para evitar overfitting
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        fp16=torch.cuda.is_available(), # Usa meia precisão se tiver GPU para ser 2x mais rápido
        report_to="none"            # Desativar relatórios externos (ex: wandb)
    )

    # 8. Treinar
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=val_tokenized,
        compute_metrics=compute_metrics,
    )

    print("🔥 A iniciar o Treino! (Isto pode demorar um pouco dependendo da GPU)...")
    trainer.train()

    # 9. Guardar o Modelo Final
    print(f"💾 A guardar o modelo final em: {MODEL_OUTPUT_DIR}")
    trainer.save_model(MODEL_OUTPUT_DIR)
    tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
    
    print("\n✅ Treino concluído com sucesso!")
    
    # Mostrar resultados finais no conjunto de validação
    print("\n📊 Métricas Finais (Validação):")
    metrics = trainer.evaluate()
    for key, value in metrics.items():
        if key.startswith("eval_"):
            print(f"   {key.replace('eval_', '')}: {value:.4f}")

if __name__ == "__main__":
    # Limpeza de memória da GPU antes de começar
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    train_student_model()

--- 🎓 FASE 5: Treino do Modelo Student (DistilBERT) ---
📂 A carregar o dataset processado...
Aviso: Dataset com 484024 registos. A realizar amostragem para 50k para treino rápido...
🔀 A dividir dados em Treino e Validação (Stratified)...
🔤 A iniciar Tokenização...


Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

🤖 A carregar modelo base (distilbert-base-uncased)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


💾 A guardar o modelo final em: F:\workspace\postech\datathon_tech_challenge_5\models\student_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Treino concluído com sucesso!

📊 Métricas Finais (Validação):


   loss: 0.0002
   accuracy: 0.9999
   f1_macro: 0.9972
   runtime: 9.5025
   samples_per_second: 1052.3600
   steps_per_second: 16.5220



[Voltar para notebook principal](./0_notebook_principal.ipynb)